# Sample Data Analysis Notebook
This notebook demonstrates various Python constructs for testing the Ember extension

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt

In [ ]:
# Define global variables
DATA_PATH = 'data/sample.csv'
OUTPUT_DIR = 'results/'
RANDOM_SEED = 42

In [ ]:
# Define a data processor class
class DataProcessor:
    def __init__(self, data_path):
        self.data_path = data_path
        self.data = None
        self.processed_data = None
    
    def load_data(self):
        """Load data from CSV file"""
        try:
            self.data = pd.read_csv(self.data_path)
            print(f"Data loaded successfully: {len(self.data)} rows")
        except FileNotFoundError:
            print("Creating sample data...")
            self.data = self._create_sample_data()
    
    def _create_sample_data(self):
        """Create sample data for testing"""
        np.random.seed(RANDOM_SEED)
        n_samples = 1000
        
        data = {
            'id': range(n_samples),
            'value': np.random.randn(n_samples),
            'category': np.random.choice(['A', 'B', 'C'], n_samples),
            'timestamp': pd.date_range(start='2024-01-01', periods=n_samples, freq='H')
        }
        return pd.DataFrame(data)
    
    def preprocess(self):
        """Preprocess the data"""
        if self.data is None:
            raise ValueError("Data not loaded")
        
        self.processed_data = self.data.copy()
        self.processed_data['value_squared'] = self.processed_data['value'] ** 2
        self.processed_data['is_positive'] = self.processed_data['value'] > 0
        
        return self.processed_data

In [ ]:
# Statistical analysis functions
def calculate_statistics(data, column='value'):
    """Calculate basic statistics for a column"""
    stats = {
        'mean': data[column].mean(),
        'std': data[column].std(),
        'min': data[column].min(),
        'max': data[column].max(),
        'median': data[column].median()
    }
    return stats

def analyze_by_category(data):
    """Analyze data grouped by category"""
    grouped = data.groupby('category')
    results = {}
    
    for category, group in grouped:
        results[category] = calculate_statistics(group)
    
    return results

In [ ]:
# Visualization class
class DataVisualizer:
    def __init__(self, figsize=(10, 6)):
        self.figsize = figsize
        plt.style.use('seaborn')
    
    def plot_distribution(self, data, column='value'):
        """Plot distribution of a column"""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=self.figsize)
        
        # Histogram
        ax1.hist(data[column], bins=30, alpha=0.7)
        ax1.set_title(f'Distribution of {column}')
        ax1.set_xlabel(column)
        ax1.set_ylabel('Frequency')
        
        # Box plot by category
        data.boxplot(column=column, by='category', ax=ax2)
        ax2.set_title(f'{column} by Category')
        
        plt.tight_layout()
        return fig
    
    def plot_timeseries(self, data):
        """Plot time series data"""
        fig, ax = plt.subplots(figsize=self.figsize)
        
        for category in data['category'].unique():
            cat_data = data[data['category'] == category]
            ax.plot(cat_data['timestamp'], cat_data['value'], 
                   label=f'Category {category}', alpha=0.7)
        
        ax.set_xlabel('Time')
        ax.set_ylabel('Value')
        ax.set_title('Time Series by Category')
        ax.legend()
        
        return fig

In [ ]:
# Main analysis pipeline
def run_analysis():
    """Run the complete analysis pipeline"""
    # Initialize processor
    processor = DataProcessor(DATA_PATH)
    processor.load_data()
    
    # Preprocess data
    processed_data = processor.preprocess()
    
    # Calculate statistics
    overall_stats = calculate_statistics(processed_data)
    category_stats = analyze_by_category(processed_data)
    
    # Visualize results
    visualizer = DataVisualizer()
    dist_fig = visualizer.plot_distribution(processed_data)
    ts_fig = visualizer.plot_timeseries(processed_data)
    
    return {
        'data': processed_data,
        'overall_stats': overall_stats,
        'category_stats': category_stats,
        'figures': [dist_fig, ts_fig]
    }

In [ ]:
# Execute analysis
if __name__ == "__main__":
    results = run_analysis()
    
    print("Analysis Complete!")
    print(f"\nOverall Statistics:")
    for key, value in results['overall_stats'].items():
        print(f"  {key}: {value:.4f}")